In [1]:
# Install kagglehub to download dataset
!pip install kagglehub

# Download the dataset
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("thedevastator/unlock-profits-with-e-commerce-sales-data")
print("Dataset path:", path)

# Load the Amazon Sale Report
import os
files = os.listdir(path)
for file in files:
    if 'Amazon' in file:
        df = pd.read_csv(os.path.join(path, file))
        print(f"Loaded: {file}")
        print(f"Shape: {df.shape}")

# Quick preview
print("\nFirst 5 rows:")
print(df.head())

Using Colab cache for faster access to the 'unlock-profits-with-e-commerce-sales-data' dataset.
Dataset path: /kaggle/input/unlock-profits-with-e-commerce-sales-data
Loaded: Amazon Sale Report.csv
Shape: (128975, 24)

First 5 rows:
   index             Order ID      Date                        Status  \
0      0  405-8078784-5731545  04-30-22                     Cancelled   
1      1  171-9198151-1101146  04-30-22  Shipped - Delivered to Buyer   
2      2  404-0687676-7273146  04-30-22                       Shipped   
3      3  403-9615377-8133951  04-30-22                     Cancelled   
4      4  407-1069790-7240320  04-30-22                       Shipped   

  Fulfilment Sales Channel  ship-service-level    Style              SKU  \
0   Merchant      Amazon.in           Standard   SET389   SET389-KR-NP-S   
1   Merchant      Amazon.in           Standard  JNE3781  JNE3781-KR-XXXL   
2     Amazon      Amazon.in          Expedited  JNE3371    JNE3371-KR-XL   
3   Merchant      Amazon.

/tmp/ipykernel_6529/2902955623.py:18: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, file))


In [2]:
# Install required libraries (if not already)
!pip install plotly nbformat

# Import libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Libraries ready!")

Libraries ready!


In [3]:
# Data cleaning
import pandas as pd
import numpy as np

# Make a copy
df_dashboard = df.copy()

# Remove cancelled orders
df_dashboard = df_dashboard[df_dashboard['Status'] != 'Cancelled']

# Convert Amount to numeric
df_dashboard['Amount'] = pd.to_numeric(df_dashboard['Amount'], errors='coerce')

# Remove missing amounts
df_dashboard = df_dashboard.dropna(subset=['Amount'])

# Convert Date
df_dashboard['Date'] = pd.to_datetime(df_dashboard['Date'], format='%m-%d-%y', errors='coerce')
df_dashboard = df_dashboard.dropna(subset=['Date'])

# Add useful columns
df_dashboard['Month'] = df_dashboard['Date'].dt.strftime('%B')
df_dashboard['YearMonth'] = df_dashboard['Date'].dt.strftime('%Y-%m')

# Calculate KPIs
total_revenue = df_dashboard['Amount'].sum()
total_orders = len(df_dashboard)
avg_order_value = total_revenue / total_orders

print("="*50)
print("DASHBOARD DATA READY")
print("="*50)
print(f"Total Revenue: ₹{total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Avg Order Value: ₹{avg_order_value:.2f}")
print(f"Date Range: {df_dashboard['Date'].min()} to {df_dashboard['Date'].max()}")
print(f"Categories: {df_dashboard['Category'].nunique()}")

DASHBOARD DATA READY
Total Revenue: ₹71,673,394.00
Total Orders: 110,414
Avg Order Value: ₹649.13
Date Range: 2022-03-31 00:00:00 to 2022-06-29 00:00:00
Categories: 9


In [6]:
import plotly.express as px
import plotly.graph_objects as go

print("="*50)
print("INTERACTIVE DASHBOARD - ECOMMERCE SALES")
print("="*50)

# Display KPIs as text
print(f"\nKEY PERFORMANCE INDICATORS")
print(f"Total Revenue: Rs.{total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value: Rs.{avg_order_value:.2f}")

# 1. Category Revenue Bar Chart
category_revenue = df_dashboard.groupby('Category')['Amount'].sum().sort_values(ascending=False).head(8)

fig1 = px.bar(x=category_revenue.values, y=category_revenue.index,
              orientation='h',
              title='Top 8 Categories by Revenue',
              labels={'x': 'Total Revenue (Rs.)', 'y': 'Category'},
              text=category_revenue.values,
              color=category_revenue.values,
              color_continuous_scale='Viridis')
fig1.update_traces(texttemplate='Rs.%{text:,.0f}', textposition='outside')
fig1.update_layout(height=500, width=900)
fig1.show()

# 2. Daily Sales Trend
daily_sales = df_dashboard.groupby('Date')['Amount'].sum().reset_index()

fig2 = px.line(daily_sales, x='Date', y='Amount',
               title='Daily Sales Trend (April - June 2022)',
               labels={'Date': 'Date', 'Amount': 'Revenue (Rs.)'})
fig2.update_layout(height=450, width=900)
fig2.show()

# 3. Average Order Value by Category
avg_category = df_dashboard.groupby('Category')['Amount'].mean().sort_values(ascending=False)

fig3 = px.bar(x=avg_category.index, y=avg_category.values,
              title='Average Order Value by Category',
              labels={'x': 'Category', 'y': 'Average Order Value (Rs.)'},
              text=avg_category.values,
              color=avg_category.values,
              color_continuous_scale='Plasma')
fig3.update_traces(texttemplate='Rs.%{text:.2f}', textposition='outside')
fig3.update_layout(height=500, width=900, xaxis_tickangle=-45)
fig3.show()

# 4. Orders by Category (Pie Chart)
category_orders = df_dashboard['Category'].value_counts().head(5)

fig4 = px.pie(values=category_orders.values, names=category_orders.index,
              title='Order Distribution by Category (Top 5)',
              hole=0.3)
fig4.update_layout(height=500, width=700)
fig4.show()

print("\nDashboard created successfully!")
print("You can hover, zoom, and interact with all charts.")

INTERACTIVE DASHBOARD - ECOMMERCE SALES

KEY PERFORMANCE INDICATORS
Total Revenue: Rs.71,673,394.00
Total Orders: 110,414
Average Order Value: Rs.649.13



Dashboard created successfully!
You can hover, zoom, and interact with all charts.


In [7]:
# Save all figures as HTML
fig1.write_html("category_revenue.html")
fig2.write_html("daily_sales_trend.html")
fig3.write_html("avg_order_value.html")
fig4.write_html("order_distribution.html")

print("Dashboard files saved:")
print("- category_revenue.html")
print("- daily_sales_trend.html")
print("- avg_order_value.html")
print("- order_distribution.html")

# Download files to your computer
from google.colab import files
files.download("category_revenue.html")
files.download("daily_sales_trend.html")
files.download("avg_order_value.html")
files.download("order_distribution.html")

Dashboard files saved:
- category_revenue.html
- daily_sales_trend.html
- avg_order_value.html
- order_distribution.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Export cleaned data to CSV
df_dashboard.to_csv("ecommerce_dashboard_data.csv", index=False)

# Download the CSV
from google.colab import files
files.download("ecommerce_dashboard_data.csv")

print("CSV file downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CSV file downloaded successfully!
